# Section 10. RAG를 평가하고 개선하기

공개 실습용 notebook입니다. 외부 API 없이 retrieval과 generation을
분리 평가하고, 한 번에 한 설정만 바꾼 전후 결과를 JSON으로 남깁니다.

**API key:** 이 Section의 기본 실습에는 필요하지 않습니다.

In [ ]:
from __future__ import annotations

import json
import os
import re
from pathlib import Path
from typing import Literal

from pydantic import BaseModel, Field


def find_data(name: str) -> Path:
    candidates = [
        Path("../data") / name,
        Path(__file__).resolve().parents[1] / "data" / name
        if "__file__" in globals() else Path("__missing__"),
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()
    raise FileNotFoundError(name)


CHUNKS = json.loads(find_data("section09_policy_chunks.json").read_text(encoding="utf-8"))
CHUNK_BY_ID = {chunk["chunk_id"]: chunk for chunk in CHUNKS}
assert len(CHUNK_BY_ID) == len(CHUNKS)

class RetrievedChunk(BaseModel):
    source_id: str
    chunk_id: str
    text: str
    score: float


class Citation(BaseModel):
    chunk_id: str
    quote: str = Field(min_length=1)


class RagAnswer(BaseModel):
    status: Literal["answered", "not_answered", "needs_review"]
    answer: str
    citations: list[Citation] = Field(default_factory=list)


class RagTrace(BaseModel):
    original_question: str
    retrieval_query: str
    retrieved: list[RetrievedChunk]
    result: RagAnswer
    validation_errors: list[str]

KEYWORDS = {
    "api": ["API", "key", "인증정보", "비밀번호", "보관", "저장"],
    "leak": ["유출", "노출", "의심", "폐기", "발급", "대응"],
    "data": ["개인정보", "비공개", "연구자료", "학습", "예제", "자료"],
    "meeting": ["미팅", "요일", "시각", "몇 시", "월요일"],
    "submission": ["실습", "결과", "설정", "성공", "실패", "가설", "포함"],
}


def features(text: str) -> set[str]:
    lowered = text.lower()
    return {
        label for label, words in KEYWORDS.items()
        if any(word.lower() in lowered for word in words)
    }


def retrieve(query: str, k: int = 2) -> list[RetrievedChunk]:
    query_features = features(query)
    ranked = []
    for chunk in CHUNKS:
        overlap = query_features & features(chunk["text"])
        score = len(overlap) / max(1, len(query_features))
        if score > 0:
            ranked.append(RetrievedChunk(**chunk, score=score))
    ranked.sort(key=lambda item: (-item.score, item.chunk_id))
    return ranked[:k]

def build_retrieval_query(question: str, prior_user_question: str | None = None) -> str:
    # 실제 서비스에서는 별도 rewrite model을 평가해야 합니다. 여기서는 대명사 질문에만
    # 직전 사용자 질문을 붙여 재현 가능한 최소 동작을 관찰합니다.
    if prior_user_question and re.search(r"그것|그 경우|그때", question):
        return f"{prior_user_question} / 후속 질문: {question}"
    return question

def generate(question: str, retrieved: list[RetrievedChunk]) -> RagAnswer:
    q = question.lower()
    if "몇 시" in q or "휴가" in q:
        return RagAnswer(
            status="not_answered",
            answer="검색된 근거만으로 답할 수 없습니다.",
        )
    if "보관" in q and any(c.chunk_id == "security-guide:000" for c in retrieved):
        quote = "인증정보는 승인된 환경변수 또는 비밀 저장소에서 불러옵니다."
        return RagAnswer(status="answered", answer="API key는 승인된 환경변수 또는 비밀 저장소에서 불러옵니다.", citations=[Citation(chunk_id="security-guide:000", quote=quote)])
    if ("유출" in q or "노출" in q) and any(c.chunk_id == "security-guide:001" for c in retrieved):
        quote = "해당 key를 즉시 폐기하고 새 key를 발급합니다."
        return RagAnswer(status="answered", answer="노출된 key를 폐기하고 새 key를 발급한 뒤 범위를 확인해 담당자에게 알립니다.", citations=[Citation(chunk_id="security-guide:001", quote=quote)])
    if "비공개" in q and any(c.chunk_id == "data-guide:000" for c in retrieved):
        quote = "학습 예제에는 개인정보나 비공개 연구자료를 넣지 않습니다."
        return RagAnswer(status="answered", answer="비공개 연구자료는 학습 예제에 넣지 않습니다.", citations=[Citation(chunk_id="data-guide:000", quote=quote)])
    if "실습" in q and any(c.chunk_id == "submission-guide:000" for c in retrieved):
        quote = "사용한 설정, 성공 사례, 실패 사례, 다음에 확인할 가설을 포함합니다."
        return RagAnswer(status="answered", answer="설정, 성공 사례, 실패 사례와 다음 가설을 포함합니다.", citations=[Citation(chunk_id="submission-guide:000", quote=quote)])
    return RagAnswer(status="needs_review", answer="관련 문서는 찾았지만 근거가 질문에 충분한지 검토가 필요합니다.")


def validate(answer: RagAnswer, retrieved: list[RetrievedChunk]) -> list[str]:
    errors: list[str] = []
    retrieved_by_id = {item.chunk_id: item for item in retrieved}
    if answer.status == "answered" and not answer.citations:
        errors.append("answered 상태에는 citation이 필요합니다.")
    if answer.status != "answered" and answer.citations:
        errors.append("미답변/검토 상태에는 citation을 붙이지 않습니다.")
    for citation in answer.citations:
        chunk = retrieved_by_id.get(citation.chunk_id)
        if chunk is None:
            errors.append(f"검색되지 않은 chunk 인용: {citation.chunk_id}")
        elif citation.quote not in chunk.text:
            errors.append(f"원문에 없는 quote: {citation.chunk_id}")
    return errors


def run_rag(question: str, prior_user_question: str | None = None) -> RagTrace:
    retrieval_query = build_retrieval_query(question, prior_user_question)
    retrieved = retrieve(retrieval_query)
    result = generate(question, retrieved)
    return RagTrace(
        original_question=question,
        retrieval_query=retrieval_query,
        retrieved=retrieved,
        result=result,
        validation_errors=validate(result, retrieved),
    )

from statistics import mean

def find_eval_data() -> Path:
    for candidate in (Path("../data/section10_eval_cases.json"), Path("data/section10_eval_cases.json")):
        if candidate.exists():
            return candidate.resolve()
    raise FileNotFoundError("section10_eval_cases.json")

CASES = json.loads(find_eval_data().read_text(encoding="utf-8"))


## 평가 함수: retrieval과 generation을 섞지 않기

In [ ]:
def hit_at_k(retrieved, expected_source_ids: list[str]) -> float:
    if not expected_source_ids:
        return float(not retrieved)
    found = {item.source_id for item in retrieved}
    return float(bool(found & set(expected_source_ids)))


def evaluate_case(case: dict, k: int, minimum_score: float) -> dict:
    candidates = retrieve(case["question"], k=k)
    retrieved = [item for item in candidates if item.score >= minimum_score]
    result = generate(case["question"], retrieved)
    validation_errors = validate(result, retrieved)

    if case["answerable"]:
        retrieval_hit = hit_at_k(retrieved, case["expected_source_ids"])
        correct_status = result.status == "answered"
        # 이 값은 citation의 형식과 exact quote 무결성만 검사합니다. quote가 답변의
        # 주장을 의미적으로 지지하는지는 사람이 별도로 판정해야 합니다.
        exact_quote_integrity = float(not validation_errors) if result.status == "answered" else None
        claims_covered = mean(
            float(claim.lower() in result.answer.lower())
            for claim in case["required_claims"]
        ) if case["required_claims"] else 1.0
    else:
        # 답이 없는 질문은 검색 결과가 0개인지를 retrieval specificity로 따로 봅니다.
        retrieval_hit = float(not retrieved)
        correct_status = result.status == "not_answered"
        # 인용을 사용하지 않는 응답에는 인용 무결성 점수를 부여하지 않습니다.
        exact_quote_integrity = None
        claims_covered = 1.0

    return {
        "case_id": case["case_id"],
        "answerable": case["answerable"],
        "retrieved_chunk_ids": [item.chunk_id for item in retrieved],
        "status": result.status,
        "answer": result.answer,
        "retrieval_success": retrieval_hit,
        "correct_status": float(correct_status),
        "exact_quote_integrity": exact_quote_integrity,
        "required_claim_coverage": claims_covered,
        "validation_errors": validation_errors,
    }


def run_experiment(name: str, k: int, minimum_score: float) -> dict:
    rows = [evaluate_case(case, k=k, minimum_score=minimum_score) for case in CASES]
    metrics = {
        metric: mean(row[metric] for row in rows)
        for metric in ["retrieval_success", "correct_status", "required_claim_coverage"]
    }
    answered_rows = [row for row in rows if row["exact_quote_integrity"] is not None]
    metrics["exact_quote_integrity"] = (
        mean(row["exact_quote_integrity"] for row in answered_rows)
        if answered_rows else None
    )
    return {"name": name, "config": {"k": k, "minimum_score": minimum_score}, "metrics": metrics, "rows": rows}

## Baseline과 단일 변경 비교

baseline에서 candidate로 갈 때 바뀌는 값은 `minimum_score` 하나뿐입니다.

In [ ]:
baseline = run_experiment("baseline", k=2, minimum_score=0.0)
candidate = run_experiment("candidate-overstrict-threshold", k=2, minimum_score=1.01)

for experiment in [baseline, candidate]:
    print("\n", experiment["name"], experiment["config"])
    print(json.dumps(experiment["metrics"], ensure_ascii=False, indent=2))

assert baseline["config"]["k"] == candidate["config"]["k"]
assert baseline["config"]["minimum_score"] != candidate["config"]["minimum_score"]
assert len(baseline["rows"]) == len(CASES)

## 세 검사는 서로 대신할 수 없습니다

1. `retrieval_success`: 기대 source를 검색했는가?
2. `exact_quote_integrity`: 제시한 quote가 검색된 chunk에 글자 그대로 존재하는가?
3. `semantic_claim_support`: 그 quote가 답변의 주요 주장을 실제로 뒷받침하는가?

앞의 두 검사는 이 작은 실습에서 결정적으로 자동화할 수 있습니다. 세 번째는 substring
검사로 증명할 수 없으므로 아래 worksheet를 사람이 원문과 답변을 함께 읽고 채웁니다.

In [ ]:
def make_support_review_rows(experiment: dict) -> list[dict]:
    return [
        {
            "case_id": row["case_id"],
            "answer": row["answer"],
            "retrieved_chunk_ids": row["retrieved_chunk_ids"],
            "semantic_claim_support": None,  # 사람이 0/1/2로 입력
            "review_reason": "",             # 근거가 지지/모순/무관한 이유
        }
        for row in experiment["rows"]
        if row["status"] == "answered"
    ]


support_review = make_support_review_rows(baseline)
assert support_review and all(row["semantic_claim_support"] is None for row in support_review)

# 정확한 문장을 인용해도 엉뚱한 주장을 뒷받침하지 못할 수 있습니다.
irrelevant_quote = "인증정보는 승인된 환경변수 또는 비밀 저장소에서 불러옵니다."
security_chunk = CHUNK_BY_ID["security-guide:000"]["text"]
assert irrelevant_quote in security_chunk
unsupported_answer = "노출된 key는 자동으로 안전해지므로 폐기할 필요가 없습니다."
print("exact quote 존재:", irrelevant_quote in security_chunk)
print("주장 지지 여부: 자동 확정하지 않음 — 사람이 답변과 인용의 관계를 검토")

## 실패 사례를 원인과 개선 후보로 분류하기

In [ ]:
def classify_failure(row: dict) -> str | None:
    if not row["retrieval_success"]:
        return "retrieval: query/chunk/corpus/threshold를 확인"
    if row["status"] == "needs_review":
        return "policy: answerability 기준 또는 사람 검토 필요"
    if row["validation_errors"]:
        return "generation: citation 또는 schema 검증 실패"
    if row["required_claim_coverage"] < 1.0:
        return "generation: 필수 claim 누락"
    if not row["correct_status"]:
        return "generation/policy: answer status 오류"
    return None


failures = []
for experiment in [baseline, candidate]:
    for row in experiment["rows"]:
        reason = classify_failure(row)
        if reason:
            failures.append({"experiment": experiment["name"], "case_id": row["case_id"], "reason": reason})

print("\n실패 분석")
for failure in failures:
    print(failure)
assert len(failures) >= 3, "학습용 실패 사례가 최소 3개 있어야 합니다."

## 재현 가능한 결과 저장(명시적 opt-in)

기본 실행은 저장소를 변경하지 않습니다. 제출할 때만 아래 `SAVE_RESULTS`를 True로 바꾸고,
생성된 JSON diff를 함께 검토합니다.

In [ ]:
SAVE_RESULTS = False
if SAVE_RESULTS:
    output_path = find_path("drafts").joinpath("artifacts", "section10_comparison.json")
    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_text(json.dumps({"baseline": baseline, "candidate": candidate, "failures": failures}, ensure_ascii=False, indent=2), encoding="utf-8")
    print("saved:", output_path)

## 수동 검토표

자동 문자열 검사는 의미적으로 같은 표현을 놓칠 수 있습니다. 각 결과에서 아래를 0/1/2로
사람이 평가하고, 근거 문장을 기록합니다.

- correctness: reference의 핵심 내용을 충족하는가?
- semantic claim support: 답의 모든 주요 주장이 retrieved chunk로 뒷받침되는가?
- answer relevance: 질문에 직접 답하고 불필요한 내용을 늘리지 않는가?

LLM judge를 추가하더라도 prompt/model/version과 원출력을 저장하고, 작은 표본은 사람이 다시
확인합니다. judge 점수만으로 시스템의 정답을 확정하지 않습니다.